This notebook shows you how to load a multi-page pharmaceutical PDF blob, tag each page with useful metadata like doc_type and page_number, and build a smart index that lets you retrieve only the content you need using filters.


In [ ]:
# ============================================
# STEP 1: Install Required Libraries
# ============================================
#
# WHAT WE'RE DOING:
# Installing LlamaIndex and its PDF reader, plus
# HuggingFace embedding models for semantic search.
#
# WHY THIS MATTERS:
# These libraries let us load PDFs, tag pages with
# metadata, and build searchable vector indexes.
#
# WHAT YOU'LL SEE:
# Installation progress (the -q flag keeps output minimal)
# ============================================

!pip install -q llama-index llama-index-readers-file llama-index-embeddings-huggingface transformers sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.9/11.9 MB 66.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 48.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 23.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 164.5/164.5 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 46.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.6/142.6 kB 10.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.


**1. Loading the pharmaceutical blob PDF**

In [ ]:
# ============================================
# STEP 2: Load the Pharmaceutical Blob PDF
# ============================================
#
# WHAT WE'RE DOING:
# Loading a multi-page pharmaceutical blob PDF.
# Each page becomes a separate Document object.
#
# WHY THIS MATTERS:
# The blob contains multiple document types
# (certificates, specs, declarations) stitched
# together. We need to process each page separately.
#
# WHAT YOU'LL SEE:
# Number of pages loaded and a preview of page 1.
# ============================================

from llama_index.readers.file import PDFReader

loader = PDFReader()
pages = loader.load_data("/content/pharma-blob-sample.pdf")  # Returns one Document per page

# Print preview
print(f"Loaded {len(pages)} pages")
print(pages[0].text[:300])

Loaded 10 pages
Cytiva
100 Results Way
Marlborough, MA 01752
United States
Page 1 / 1
cytiva.com
3 June, 2022
Re: AKTA ready Flow Kit Storage Conditions
To Whom It May Concern,
The recommended storage temperature for standard AKTA ready flow kits is provided in Section 8.3 of the Operating
Instructions 28960345 and


**2. Adding metadata like doc_type and page_number**

In [ ]:
# ============================================
# STEP 3: Add Page-Level Metadata
# ============================================
#
# WHAT WE'RE DOING:
# Attaching metadata (page number, source file)
# to each page Document object.
#
# WHY THIS MATTERS:
# Metadata lets us trace results back to their
# source page and filter by document attributes.
#
# WHAT YOU'LL SEE:
# No output (metadata is attached silently).
# ============================================

documents = []
for i, doc in enumerate(pages):
    doc.metadata = {
        "page_number": i + 1,
        "source_file": "pharma-blob-sample.pdf"
    }
    documents.append(doc)

In [ ]:
# ============================================
# STEP 4: Assign Document Types
# ============================================
#
# WHAT WE'RE DOING:
# Labeling each page with its pharmaceutical
# document type. In a real pipeline, an LLM would
# classify these automatically (see Notebook 1).
#
# WHY THIS MATTERS:
# doc_type metadata enables filtered retrieval -
# you can search only within specific document types.
#
# WHAT YOU'LL SEE:
# No output (doc_type is assigned silently).
# ============================================

doc_type_array = [
    "Cover Letter",            # Page 1
    "Certificate of Quality",  # Page 2
    "Certificate of Quality",  # Page 3
    "Packaging Specification", # Page 4
    "Packaging Specification", # Page 5
    "BSE/TSE Declaration",     # Page 6
    "Material Description",    # Page 7
    "Supplier Qualification",  # Page 8
    "Supplier Qualification",  # Page 9
    "Chain of Custody",        # Page 10
]

for doc, doc_type in zip(documents, doc_type_array):
    doc.metadata["doc_type"] = doc_type

In [ ]:
# ============================================
# STEP 5: Preview Metadata
# ============================================
#
# WHAT WE'RE DOING:
# Printing the metadata and a text preview for
# the first two pages to verify tagging worked.
#
# WHY THIS MATTERS:
# Always verify your metadata before indexing.
# Wrong labels lead to wrong retrieval results.
#
# WHAT YOU'LL SEE:
# Metadata dict and first 150 chars for pages 1-2.
# ============================================

for doc in documents[:2]:
    print(doc.metadata)
    print(doc.text[:150], "\n---\n")

{'page_number': 1, 'source_file': 'pharma-blob-sample.pdf', 'doc_type': 'Cover Letter'}
Cytiva
100 Results Way
Marlborough, MA 01752
United States
Page 1 / 1
cytiva.com
3 June, 2022
Re: AKTA ready Flow Kit Storage Conditions
To Whom It Ma 
---

{'page_number': 2, 'source_file': 'pharma-blob-sample.pdf', 'doc_type': 'Certificate of Quality'}
Certificate of Quality
This product is manufactured in compliance with our ISO 9001 certified quality management system.
Issued by Cytiva Westborough  
---



**3. Indexing and retrieving with metadata filters**

In [ ]:
# ============================================
# STEP 6: Build Vector Index with Metadata
# ============================================
#
# WHAT WE'RE DOING:
# Converting all pages into vector embeddings
# and storing them in a searchable index.
# The metadata (doc_type, page_number) travels
# with each vector.
#
# WHY THIS MATTERS:
# The index enables semantic search. Metadata
# enables filtered search (e.g., only search
# within "Certificate of Quality" pages).
#
# WHAT YOU'LL SEE:
# Embedding model download (first run) and
# confirmation of index creation.
# ============================================

from llama_index.core import VectorStoreIndex
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

# Use a local embedding model
embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5")

# Build a vector index enriched with metadata (doc_type, page_number, source_file)
index = VectorStoreIndex.from_documents(documents, embed_model=embed_model)

print("Metadata-enriched index created with", len(documents), "documents.")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Metadata-enriched index created with 10 documents.


In [ ]:
# ============================================
# STEP 7: Query with Metadata Filtering
# ============================================
#
# WHAT WE'RE DOING:
# Querying the index, then filtering results
# to only show chunks from a specific doc type.
#
# WHY THIS MATTERS:
# Without filtering, the retriever might return
# chunks from irrelevant document types. Metadata
# filtering narrows results to the right documents.
#
# WHAT YOU'LL SEE:
# Only results from "BSE/TSE Declaration" pages,
# even though the query searches the full index.
# ============================================

# Query the full index
retriever = index.as_retriever()
query = "What are the BSE/TSE compliance dates for this product?"
all_results = retriever.retrieve(query)

# Filter results using metadata
filtered_results = [
    r for r in all_results if r.metadata.get("doc_type") == "BSE/TSE Declaration"
]

# Display the filtered results
for i, r in enumerate(filtered_results):
    print(f"--- Result {i+1} ---")
    print(r.text[:500])
    print("Metadata:", r.metadata)
    print("\n")

--- Result 1 ---
Declaration Regarding Transmissible Spongiform
Encephalopathies (BSE/TSE Compliance Statement)
Manufacturer: Cytiva Sweden AB
Address: Bjorkgatan 30, SE-751 84 Uppsala, Sweden
Product Family: AKTA ready Flow Kits
Applicable Part Numbers: 29477427, 29184612, 28930182, 28930183
Declaration
We hereby declare that the above-referenced products do not contain any materials of animal origin, including but not
limited to bovine, ovine, caprine, or porcine derived materials.
All polymeric materials used
Metadata: {'page_number': 6, 'source_file': 'pharma-blob-sample.pdf', 'doc_type': 'BSE/TSE Declaration'}


